In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
import re
import random
import math
import time
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


#  Data cleaning and loading
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zа-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_data(filepath, num_examples=30000):
    eng_sentences = []
    rus_sentences = []

    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[:num_examples]:
            parts = line.split('\t')
            if len(parts) >= 2:
                eng_sentences.append(clean_text(parts[0]))
                rus_sentences.append(clean_text(parts[1]))

    return eng_sentences, rus_sentences


eng_data, rus_data = load_data(r'C:\Users\srbuh\PycharmProjects\nlp_course_2026_2\Srbuhi Pupuyan\Lab6\rus.txt', 100000)

print(f"Count of sentences: {len(eng_data)}")
print("Example (Eng):", eng_data[100])
print("Example (Rus):", rus_data[100])


# Vocabulary
class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.word2idx = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.idx2word = {0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>'}
        self.word_freq = Counter()
        self.idx = 4

    def add_sentence(self, sentence):
        for word in sentence.split():
            self.word_freq[word] += 1
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def __len__(self):
        return len(self.word2idx)


# Build vocabularies
eng_vocab = Vocabulary("English")
rus_vocab = Vocabulary("Russian")

for eng, rus in zip(eng_data, rus_data):
    eng_vocab.add_sentence(eng)
    rus_vocab.add_sentence(rus)

print(f"English vocabulary size: {len(eng_vocab)}")
print(f"Russian vocabulary size: {len(rus_vocab)}")


# Dataset and Collate function
class TranslationDataset(Dataset):
    def __init__(self, eng_sentences, rus_sentences, eng_vocab, rus_vocab):
        self.eng_sentences = eng_sentences
        self.rus_sentences = rus_sentences
        self.eng_vocab = eng_vocab
        self.rus_vocab = rus_vocab

    def __len__(self):
        return len(self.eng_sentences)

    def __getitem__(self, i):
        eng_sentence = self.eng_sentences[i]
        rus_sentence = self.rus_sentences[i]

        eng_tensor = [self.eng_vocab.word2idx.get(w, 3) for w in eng_sentence.split()]

        # Add <sos> (1) at the start and <eos> (2) at the end
        rus_tensor = [1] + [self.rus_vocab.word2idx.get(w, 3) for w in rus_sentence.split()] + [2]

        return torch.tensor(eng_tensor), torch.tensor(rus_tensor)


def collate_fn(batch):
    eng_batch, rus_batch = [], []
    for eng_item, rus_item in batch:
        eng_batch.append(eng_item)
        rus_batch.append(rus_item)

    # Pad sequences to the same length within the batch
    eng_batch = pad_sequence(eng_batch, padding_value=0, batch_first=False)
    rus_batch = pad_sequence(rus_batch, padding_value=0, batch_first=False)

    return eng_batch, rus_batch


# Split data: 80% train, 20% validation
eng_train, eng_val, rus_train, rus_val = train_test_split(
    eng_data, rus_data, test_size=0.2, random_state=42
)

train_dataset = TranslationDataset(eng_train, rus_train, eng_vocab, rus_vocab)
val_dataset = TranslationDataset(eng_val, rus_val, eng_vocab, rus_vocab)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)

for src, trg in train_loader:
    print("Source (English) tensor shape [SeqLen, BatchSize]:", src.shape)
    print("Target (Russian) tensor shape [SeqLen, BatchSize]:", trg.shape)
    break

Device: cuda
Count of sentences: 100000
Example (Eng): he ran
Example (Rus): он бежал
English vocabulary size: 7216
Russian vocabulary size: 20764
Source (English) tensor shape [SeqLen, BatchSize]: torch.Size([6, 128])
Target (Russian) tensor shape [SeqLen, BatchSize]: torch.Size([9, 128])


In [5]:
#Model Architecture

# 1. ENCODER
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout_rate=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.dropout = nn.Dropout(dropout_rate)
        # Return all encoder outputs so attention can attend over them
        self.lstm = nn.LSTM(emb_dim, hid_dim, num_layers=2, dropout=dropout_rate)

    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.dropout(self.embedding(src))

        # outputs: [src_len, batch_size, hid_dim]  ← used by attention
        # hidden, cell: [num_layers, batch_size, hid_dim]
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell


# 2. ATTENTION
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v    = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [batch_size, hid_dim]  (decoder top-layer hidden)
        # encoder_outputs: [src_len, batch_size, hid_dim]

        src_len = encoder_outputs.shape[0]

        # Repeat decoder hidden across every source position
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)

        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        # encoder_outputs: [batch_size, src_len, hid_dim]

        # Compute energy scores
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return F.softmax(attention, dim=1)

# 3. DECODER
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout_rate=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.attention  = Attention(hid_dim)

        self.embedding = nn.Embedding(output_dim, emb_dim)
        # LSTM input = embedding + context vector (hid_dim)
        self.lstm   = nn.LSTM(emb_dim + hid_dim, hid_dim, num_layers=2, dropout=dropout_rate)
        self.fc_out = nn.Linear(emb_dim + hid_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, input, hidden, cell, encoder_outputs):
        # input: [batch_size]
        input    = input.unsqueeze(0)                     # [1, batch_size]
        embedded = self.dropout(self.embedding(input))    # [1, batch_size, emb_dim]

        #Attention
        top_hidden = hidden[-1]                           # [batch_size, hid_dim]
        a = self.attention(top_hidden, encoder_outputs)   # [batch_size, src_len]
        a = a.unsqueeze(1)                                # [batch_size, 1, src_len]

        enc_out_perm = encoder_outputs.permute(1, 0, 2)  # [batch_size, src_len, hid_dim]

        # Context vector: weighted sum of encoder outputs
        weighted = torch.bmm(a, enc_out_perm)             # [batch_size, 1, hid_dim]
        weighted = weighted.permute(1, 0, 2)              # [1, batch_size, hid_dim]

        # Concatenate embedding and context vector as LSTM input
        rnn_input = torch.cat((embedded, weighted), dim=2)

        output, (hidden, cell) = self.lstm(rnn_input, (hidden, cell))

        embedded = embedded.squeeze(0)   # [batch_size, emb_dim]
        output   = output.squeeze(0)     # [batch_size, hid_dim]
        weighted = weighted.squeeze(0)   # [batch_size, hid_dim]

        # Predict next token from [output || context || embedding]
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))

        return prediction, hidden, cell


# 4. SEQ2SEQ
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size     = src.shape[1]
        trg_len        = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)

        # First decoder input is the <sos> token
        input = trg[0, :]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[t] = output

            top1 = output.argmax(1)
            teacher_force = random.random() < teacher_forcing_ratio
            input = trg[t] if teacher_force else top1

        return outputs


In [6]:
#Hyperparameters and model initialisation
INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(rus_vocab)
EMB_DIM = 256
HID_DIM = 1024
ENC_DROPOUT = 0.2
DEC_DROPOUT = 0.2

enc = Encoder(INPUT_DIM, EMB_DIM, HID_DIM, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, EMB_DIM, HID_DIM, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

PAD_IDX = rus_vocab.word2idx['<pad>']
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f'The model has {count_parameters(model):,} trainable parameters')


#Training and Evaluation functions

def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0

    for src, trg in loader:
        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)
        # output: [trg_len, batch_size, output_dim]
        # Reshape for CrossEntropyLoss: ignore the first <sos> timestep
        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim)
        trg = trg[1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in loader:
            src = src.to(device)
            trg = trg.to(device)

            # Turn off teacher forcing during evaluation
            output = model(src, trg, teacher_forcing_ratio=0.0)

            output_dim = output.shape[-1]
            output = output[1:].reshape(-1, output_dim)
            trg = trg[1:].reshape(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(loader)


def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    return int(elapsed / 60), int(elapsed % 60)


#  Training loop
N_EPOCHS = 30
CLIP = 1.0
MODEL_PATH = 'seq2seq_attention_best_model.pt'

best_valid_loss = float('inf')

print("Starting training...\n")
for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, val_loader, criterion)

    end_time = time.time()
    mins, secs = epoch_time(start_time, end_time)

    # Save the model whenever validation loss improves
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), MODEL_PATH)

    print(f'Epoch: {epoch + 1:02} | Time: {mins}m {secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')
#Load best model weights
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
print(f"Best model loaded from '{MODEL_PATH}'")


The model has 88,613,148 trainable parameters
Starting training...

Epoch: 01 | Time: 2m 18s
	Train Loss: 3.630 | Train PPL:  37.706
	 Val. Loss: 2.661 |  Val. PPL:  14.315
Epoch: 02 | Time: 2m 16s
	Train Loss: 1.989 | Train PPL:   7.311
	 Val. Loss: 2.180 |  Val. PPL:   8.850
Epoch: 03 | Time: 2m 16s
	Train Loss: 1.426 | Train PPL:   4.163
	 Val. Loss: 2.045 |  Val. PPL:   7.728
Epoch: 04 | Time: 2m 16s
	Train Loss: 1.162 | Train PPL:   3.195
	 Val. Loss: 2.018 |  Val. PPL:   7.521
Epoch: 05 | Time: 2m 16s
	Train Loss: 1.017 | Train PPL:   2.765
	 Val. Loss: 2.005 |  Val. PPL:   7.428
Epoch: 06 | Time: 2m 20s
	Train Loss: 0.927 | Train PPL:   2.527
	 Val. Loss: 2.022 |  Val. PPL:   7.552
Epoch: 07 | Time: 2m 19s
	Train Loss: 0.862 | Train PPL:   2.368
	 Val. Loss: 2.036 |  Val. PPL:   7.658
Epoch: 08 | Time: 2m 16s
	Train Loss: 0.824 | Train PPL:   2.281
	 Val. Loss: 2.022 |  Val. PPL:   7.556
Epoch: 09 | Time: 2m 15s
	Train Loss: 0.788 | Train PPL:   2.200
	 Val. Loss: 2.022 |  Val. 

In [7]:
#Greedy inference
def translate_sentence_attention(sentence, eng_vocab, rus_vocab, model, device, max_len=50):
    model.eval()

    if isinstance(sentence, str):
        sentence = clean_text(sentence)
        tokens   = sentence.split()
    else:
        tokens = sentence

    text_to_indices = [eng_vocab.word2idx.get(token, eng_vocab.word2idx['<unk>']) for token in tokens]
    sentence_tensor = torch.LongTensor(text_to_indices).unsqueeze(1).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(sentence_tensor)

    outputs = [rus_vocab.word2idx['<sos>']]

    for _ in range(max_len):
        previous_word = torch.LongTensor([outputs[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(previous_word, hidden, cell, encoder_outputs)
            best_guess = output.argmax(1).item()

        outputs.append(best_guess)

        if best_guess == rus_vocab.word2idx['<eos>']:
            break

    translated_sentence = [rus_vocab.idx2word[idx] for idx in outputs]
    return " ".join(translated_sentence[1:-1])


# Quick sanity check
test_sentences = [
    "i am cold",
    "he is a good boy",
    "how are you",
    "run away"
]

print("--- Testing Translations (Greedy + Attention) ---")
for sentence in test_sentences:
    translation = translate_sentence_attention(sentence, eng_vocab, rus_vocab, model, device)
    print(f"English: {sentence}")
    print(f"Russian: {translation}")
    print("-" * 30)

--- Testing Translations (Greedy + Attention) ---
English: i am cold
Russian: я замёрзла
------------------------------
English: he is a good boy
Russian: он хороший мальчик
------------------------------
English: how are you
Russian: как дела как дела
------------------------------
English: run away
Russian: беги убежать
------------------------------


In [8]:
#N-grams & Clipped Precision
def get_ngrams(tokens, n):
    """Return a Counter of all n-grams in the token list."""
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngram_slice = tuple(tokens[i : i + n])
        ngrams.append(ngram_slice)
    return Counter(ngrams)


def clipped_precision(hypothesis, reference, n):
    hyp_ngrams = get_ngrams(hypothesis, n)
    ref_ngrams = get_ngrams(reference,  n)

    total_hyp_ngrams = sum(hyp_ngrams.values())
    if total_hyp_ngrams == 0:
        return 0.0

    clipped_count = 0
    for ngram, count in hyp_ngrams.items():
        # A word can only be credited as many times as it appears in the reference
        clipped_count += min(count, ref_ngrams[ngram])

    return clipped_count / total_hyp_ngrams

In [9]:
# Brevity Penalty & BLEU Score
def brevity_penalty(hypothesis, reference):
    c = len(hypothesis)
    r = len(reference)

    if c >= r:
        return 1.0
    else:
        return math.exp(1 - (r / c))


def bleu_score(hypothesis, reference):
    hyp_tokens = hypothesis.split()
    ref_tokens = reference.split()

    precisions = []
    for n in range(1, 5):
        p = clipped_precision(hyp_tokens, ref_tokens, n)
        if p == 0:
            return 0.0
        precisions.append(p)

    # Geometric mean in log space (uniform weight 1/4 per n-gram order)
    log_sum = sum(math.log(p) for p in precisions)
    geometric_mean = math.exp(log_sum / 4.0)

    bp = brevity_penalty(hyp_tokens, ref_tokens)
    return bp * geometric_mean


# Sanity check against the known reference value (~0.516)
hyp_text = "the match was postponed because of the snow"
ref_text = "the match was postponed because it was snowing"

final_bleu = bleu_score(hyp_text, ref_text)
print(f"Hypothesis: {hyp_text}")
print(f"Reference:  {ref_text}")
print(f"Final BLEU Score (expected ~0.516): {final_bleu:.4f}")

Hypothesis: the match was postponed because of the snow
Reference:  the match was postponed because it was snowing
Final BLEU Score (expected ~0.516): 0.5170


In [10]:
#Beam Search Decoding (Attention-aware)
def beam_search_attention(sentence, eng_vocab, rus_vocab, model, device, beam_width=3, max_len=50):
    model.eval()

    # Prepare input
    if isinstance(sentence, str):
        sentence = clean_text(sentence)
        tokens = sentence.split()
    else:
        tokens = sentence

    text_to_indices = [eng_vocab.word2idx.get(token, eng_vocab.word2idx['<unk>']) for token in tokens]
    sentence_tensor = torch.LongTensor(text_to_indices).unsqueeze(1).to(device)

    # Encode the source sentence once; share encoder_outputs across all beams
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(sentence_tensor)

    sos_idx = rus_vocab.word2idx['<sos>']
    eos_idx = rus_vocab.word2idx['<eos>']

    # Each beam entry: (score, token_sequence, hidden, cell)
    beam = [(0.0, [sos_idx], hidden, cell)]
    completed = []

    for _ in range(max_len):
        new_beam = []

        for score, seq, h, c in beam:
            # If this sequence already ended, move it to completed
            if seq[-1] == eos_idx:
                completed.append((score, seq))
                continue

            previous_word = torch.LongTensor([seq[-1]]).to(device)

            with torch.no_grad():
                output, next_h, next_c = model.decoder(
                    previous_word, h, c, encoder_outputs
                )
                # Convert logits to log-probabilities for numerical stability
                log_probs = F.log_softmax(output.squeeze(0), dim=0)

            # Expand each candidate by the top beam_width next tokens
            top_log_probs, top_indices = log_probs.topk(beam_width)

            for i in range(beam_width):
                next_word = top_indices[i].item()
                log_prob = top_log_probs[i].item()

                new_score = score + log_prob
                new_seq = seq + [next_word]

                new_beam.append((new_score, new_seq, next_h, next_c))

        if not new_beam:
            break

        # Keep only the top beam_width candidates (prune the beam)
        new_beam.sort(key=lambda x: x[0], reverse=True)
        beam = new_beam[:beam_width]

    # Move remaining live candidates into completed
    completed.extend([(score, seq) for score, seq, _, _ in beam])

    if not completed:
        completed = [(score, seq) for score, seq, _, _ in beam]

    # Pick the highest-scoring completed sequence
    completed.sort(key=lambda x: x[0], reverse=True)
    best_score, best_seq = completed[0]

    # Decode token indices back to words (strip <sos> and <eos>)
    translated_words = []
    for idx in best_seq:
        if idx == sos_idx: continue
        if idx == eos_idx: break
        translated_words.append(rus_vocab.idx2word[idx])

    return " ".join(translated_words)
# Quick comparison: Greedy vs Beam Search
test_sentence = "i am cold"
print(f"Original:          {test_sentence}")

greedy_out = translate_sentence_attention(test_sentence, eng_vocab, rus_vocab, model, device)
print(f"Greedy Output:     {greedy_out}")

beam_out = beam_search_attention(test_sentence, eng_vocab, rus_vocab, model, device, beam_width=5)
print(f"Beam (k=5) Output: {beam_out}")

Original:          i am cold
Greedy Output:     я замёрзла
Beam (k=5) Output: я замёрзла


In [11]:
# Experiment & Final Report
NUM_SENTENCES = 100
test_pairs = list(zip(eng_val[:NUM_SENTENCES], rus_val[:NUM_SENTENCES]))

# Define the decoding modes to compare
modes = [
    {
        "name": "Greedy",
        "func": lambda src: translate_sentence_attention(src, eng_vocab, rus_vocab, model, device),
        "width": None
    },
    {
        "name": "Beam (w=3)",
        "func": lambda src: beam_search_attention(src, eng_vocab, rus_vocab, model, device, beam_width=3),
        "width": 3
    },
    {
        "name": "Beam (w=5)",
        "func": lambda src: beam_search_attention(src, eng_vocab, rus_vocab, model, device, beam_width=5),
        "width": 5
    },
    {
        "name": "Beam (w=10)",
        "func": lambda src: beam_search_attention(src, eng_vocab, rus_vocab, model, device, beam_width=10),
        "width": 10
    }
]

print(f"Running experiment on {NUM_SENTENCES} sentences...\n")

results_table = []
all_translations = {mode["name"]: [] for mode in modes}

for mode in modes:
    name = mode["name"]
    func = mode["func"]

    total_time = 0.0
    total_bleu = 0.0

    for src, trg in test_pairs:
        start_time = time.time()
        pred = func(src)
        end_time = time.time()

        total_time += end_time - start_time
        total_bleu += bleu_score(pred, trg)

        all_translations[name].append(pred)

    avg_time = total_time / NUM_SENTENCES
    avg_bleu = total_bleu / NUM_SENTENCES
    results_table.append((name, avg_bleu, avg_time))

# Print summary table
print("=== EXPERIMENT RESULTS (Attention Model) ===")
print(f"{'Decoding Method':<15} | {'Average BLEU-4':<15} | {'Time per sentence (s)':<25}")
print("-" * 60)
for name, bleu, t in results_table:
    print(f"{name:<15} | {bleu:<15.4f} | {t:<25.4f}")
print()

# Print 5 concrete side-by-side examples
print("=== 5 CONCRETE EXAMPLES (Greedy vs Beam w=5) ===")
example_indices = [5, 20, 45, 70, 95]

for idx in example_indices:
    src, trg = test_pairs[idx]
    greedy_pred = all_translations["Greedy"][idx]
    beam5_pred = all_translations["Beam (w=5)"][idx]

    print(f"Source (EN):    {src}")
    print(f"Reference (RU): {trg}")
    print(f"Greedy Out:     {greedy_pred}")
    print(f"Beam (w=5) Out: {beam5_pred}")
    print("-" * 50)

Running experiment on 100 sentences...

=== EXPERIMENT RESULTS (Attention Model) ===
Decoding Method | Average BLEU-4  | Time per sentence (s)    
------------------------------------------------------------
Greedy          | 0.0681          | 0.0108                   
Beam (w=3)      | 0.0615          | 0.1037                   
Beam (w=5)      | 0.0615          | 0.2208                   
Beam (w=10)     | 0.0615          | 0.5551                   

=== 5 CONCRETE EXAMPLES (Greedy vs Beam w=5) ===
Source (EN):    i used to admire tom
Reference (RU): я раньше восхищался томом
Greedy Out:     я раньше раньше уважал
Beam (w=5) Out: я раньше раньше уважал
--------------------------------------------------
Source (EN):    just dont tell tom
Reference (RU): только не говорите тому
Greedy Out:     только не говори тому
Beam (w=5) Out: только не говори тому
--------------------------------------------------
Source (EN):    our tv isnt working
Reference (RU): у нас телевизор не работает
Gree